导入包  
request：获取网页内容  
beautifulSoup：处理网页内容  
tqdm：进度条  

In [1]:
import datetime
import os
import re
import cdflib
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
import re
from time import sleep
import json

下载函数  
获取网址中的所有链接 def search_url  
链接下载，附带报错处理  
错误时不写入文件

In [2]:
def search_url(url,filename):

    response = requests.get(url)
    html_content = response.text

    soup = BeautifulSoup(html_content, 'html.parser')

    file_urls = []
    file_elements = soup.find_all('a', href=True)  # 找到所有带有href属性的<a>元

    for element in file_elements:
        file_url = element['href']
        file_urls.append(file_url)
    urls=[string for string in file_urls if re.match(filename, string)]
    response.close()
    return urls
def requests_downlaod(url,save_path):
    try:
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            total_size = int(response.headers.get("content-length", 0))
            block_size = 1024
            progress_bar = tqdm(total=total_size, unit="B", unit_scale=True)
            buffer = bytearray()  # 创建字节缓冲区
            for data in response.iter_content(block_size):
                progress_bar.update(len(data))
                buffer.extend(data)  # 将下载的数据添加到缓冲区
            progress_bar.close()
            
            # 将缓冲区中的数据写入文件
            with open(save_path, "wb") as file:
                file.write(buffer)
            return True
        response.close()
    except requests.exceptions.RequestException as e:
        print(" url error: "+str(e)+',sleep 60s')
        sleep(60)
        return False

下载方式：使用字典记录逻辑地址

In [3]:
def downlaod_model(model):
    url_path_0='https://lasp.colorado.edu/maven/sdc/public/data/sci/'
    with open("data_format.json", "r") as file:
        data = json.load(file)
    save_dir = data["save_dir"]
    data_model = data["data_model"]
    model_key = {}
    for key, value in data_model.items():
        model_key[key] = value[:3]
    # model_key={
    #     'SWEA_spec':    ['swe/l2/',r'mvn_swe_l2_svyspec_.+\.cdf',"SWEA/svyspec"],
    #     'LPW_mrgscpot': ['lpw/l2/',r"mvn_lpw_l2_mrgscpot_.+\.cdf","LPW/mrgscpot"],
    #     'SWEA_pad':     ['swe/l2/',r'mvn_swe_l2_arcpad_.+\.cdf',"SWEA/arcpad"],
    #     'SWEA_pad_svy': ['swe/l2/',r'mvn_swe_l2_svypad_.+\.cdf',"SWEA/svypad"],
    #     'LPW_wave':     ['lpw/l2/',r'mvn_lpw_l2_wspecpas_.+\.cdf',"LPW/wspecpas"],
    #     'MAG_ss':       ['mag/l2/',r"mvn_mag_l2_\d{7}ss_\d{8}_v\d{2}_r\d{2}\.sts",'MAG/ss'],
    #     'MAG_ss1s':     ['mag/l2/',r"mvn_mag_l2_\d{7}ss1s_\d{8}_v\d{2}_r\d{2}\.sts",'MAG/ss_1s'],
    #     'MAG_pc':       ['mag/l2/',r"mvn_mag_l2_\d{7}pc_\d{8}_v\d{2}_r\d{2}\.sts",'MAG/pc'],
    #     'MAG_pc1s':     ['mag/l2/',r"mvn_mag_l2_\d{7}pc1s_\d{8}_v\d{2}_r\d{2}\.sts",'MAG/pc_1s'],
    #     "STATIC":       ['sta/l2/',r"mvn_sta_l2_c6-32e64m_.+\.cdf","STATIC/c6-32e64m"],
    #     'KP':           ['kp/insitu/',r'^mvn_kp_insitu_.+\.tab$','KP'],
    #     'LPW_lpnt':     ['lpw/l2/',r"mvn_lpw_l2_lpnt_.+\.cdf","LPW/lpnt"],
    #     'LPW_wn':       ['lpw/l2/',r"mvn_lpw_l2_wn_.+\.cdf","LPW/wn"],
    #     }

    model_url,filename,pathname=model_key[model]
    url = url_path_0+model_url
    save_path =  save_dir+pathname
    return [url,save_path,filename]
def find_downloaded_file(file_names, element):
    for file_name in file_names:
        if element in file_name:
            return True
    return False
def file_check(file_names,save_path,model):
    if model == 'cdf':
        for f in file_names:
            try :
                data=cdflib.cdfread.CDF(save_path+f)
                print(data)
                print(f,'true')
                continue
            except Exception as e:
                os.remove(os.path.join(save_path,f))
                print('delete:' ,f,e)

主程序  
获取所有已经存在的文件  

In [4]:
start_date   = datetime.date(2015, 5, 1)
end_date     = datetime.date(2023, 8, 1)

current_date = start_date

models=['SWEA_spec','SWEA_pad','LPW_wave','MAG_ss','MAG_ss1s',"STATIC","KP",'LPW_mrgscpot','SWEA_pad_svy','LPW_lpnt']
model='MAG_ss1s'#'SWEA_pad_svy'#'LPW_wn' #'SWEA_pad_svy'  #'MAG_pc1s'#'MAG'

url_path,save_path,filestyle = downlaod_model(model)
file_names = os.listdir(save_path)
#file_check(file_names,save_path,'cdf')
with open("E:/MAVEN/lists/"+model+'_list.txt', 'w') as file:
    for item in file_names:
        file.write(str(item) + '\n') 

循环所有日期  
跳过已下载文件  

In [5]:
#get all url:
while current_date <= end_date:
    date=str(current_date.strftime("%Y%m%d"))

    if find_downloaded_file(file_names, date):
        print(date,model,'skip')
        start_date=current_date
        current_date+=datetime.timedelta(days=1)
        continue
    year=current_date.year
    month=current_date.month
    yyyymm=str(current_date.strftime("%Y/%m/"))
    urls    = search_url(url_path+yyyymm,filestyle)
    print(len(urls),'urls in ', yyyymm)
    for url in urls:
        filename = str(url)
        date=re.findall(r"\d{8}", filename)
        if find_downloaded_file(file_names, filename):
            print(date,model,'exist')
            continue
        logic = requests_downlaod(url_path+yyyymm+filename,save_path+filename)
        if logic == False: 
            os.remove(os.path.join(save_path,filename))
            print('remove',date)
        print('\n',date,model,logic,end='\n')
    month+=1
    if month == 13:
        month=1
        year+=1
    current_date = datetime.date(year, month, 1)

20150501 MAG_ss1s skip
20150502 MAG_ss1s skip
20150503 MAG_ss1s skip
20150504 MAG_ss1s skip
20150505 MAG_ss1s skip
20150506 MAG_ss1s skip
20150507 MAG_ss1s skip
20150508 MAG_ss1s skip
20150509 MAG_ss1s skip
20150510 MAG_ss1s skip
20150511 MAG_ss1s skip
20150512 MAG_ss1s skip
20150513 MAG_ss1s skip
20150514 MAG_ss1s skip
20150515 MAG_ss1s skip
20150516 MAG_ss1s skip
20150517 MAG_ss1s skip
20150518 MAG_ss1s skip
20150519 MAG_ss1s skip
20150520 MAG_ss1s skip
20150521 MAG_ss1s skip
20150522 MAG_ss1s skip
20150523 MAG_ss1s skip
20150524 MAG_ss1s skip
20150525 MAG_ss1s skip
20150526 MAG_ss1s skip
20150527 MAG_ss1s skip
20150528 MAG_ss1s skip
20150529 MAG_ss1s skip
20150530 MAG_ss1s skip
20150531 MAG_ss1s skip
20150601 MAG_ss1s skip
20150602 MAG_ss1s skip
20150603 MAG_ss1s skip
20150604 MAG_ss1s skip
20150605 MAG_ss1s skip
20150606 MAG_ss1s skip
20150607 MAG_ss1s skip
20150608 MAG_ss1s skip
20150609 MAG_ss1s skip
20150610 MAG_ss1s skip
20150611 MAG_ss1s skip
20150612 MAG_ss1s skip
20150613 MA

100%|██████████| 13.0M/13.0M [00:03<00:00, 4.31MB/s]



 ['20230515'] MAG_ss1s True


100%|██████████| 13.0M/13.0M [00:03<00:00, 4.06MB/s]



 ['20230516'] MAG_ss1s True


...


 ['20230518'] MAG_ss1s True


  4%|▍         | 575k/13.0M [01:57<1:40:05, 2.06kB/s] 

 url error: ('Connection broken: IncompleteRead(576124 bytes read, 12395087 more expected)', IncompleteRead(576124 bytes read, 12395087 more expected)),sleep 60s


  4%|▍         | 575k/13.0M [09:37<3:27:24, 996B/s]  


FileNotFoundError: [WinError 2] 系统找不到指定的文件。: 'E:/MAVEN/MAG/ss_1s/mvn_mag_l2_2023139ss1s_20230519_v01_r01.sts'

输出list文件记录所有已下载文件

In [ ]:
file_names = os.listdir(save_path)
with open("E:/MAVEN/lists/"+model+'_list.txt', 'w') as file:
    for item in file_names:
        file.write(str(item) + '\n')